In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..')) 
from src.utils.config import USERNAME, ORGNAME
from src.utils.redivis_client import RedivisCatalog, print_redivis_tree
from src.etl.ingest import fetch_hcup
from src.etl.transform import enrich_zip_data

import redivis

# ==========================================
# WARNING: INTERACTIVE AUTHENTICATION
# ==========================================
# If no REDIVIS_API_TOKEN is set in your environment, Redivis will pause 
# execution here and output a URL in the terminal.
# You must click the URL and approve access via your web browser.
# 
# Upon success, Redivis securely caches your credentials locally in:
# Mac/Linux: ~/.redivis/python_credentials
# Windows:   C:\Users\YourUser\.redivis\python_credentials
#
# Future runs on this specific PC will use that hidden cache silently 
# without prompting you again, until the token expires or the folder is deleted.

# Trigger the authentication flow
print(f"Authenticating as `{USERNAME}`...\n")
if redivis.organization(ORGNAME).exists():  #  # <--- This forces to wait for authentication here
    print(f"\033[92m\u2714\033[0m Redivis Python Client `{USERNAME}` successfully authenticated")

In [ ]:
# hcup = RedivisCatalog(ORGNAME)

# Call as a property, not a function
#display(hcup.datasets)

#display(hcup.get_tables("nevada_hcup:4nca:v4_1"))

#display(hcup.get_variables("nevada_hcup:4nca:v4_1","nv_sedd_2014_core:242v"))



In [ ]:
#print_redivis_tree(ORGNAME, limit_tables=0) # default: limit_tables=3

In [ ]:
#SEDD 

selected_cntys = {
    'Arizona': ('Pima, AZ', 'Pinal, AZ', 'Yavapai, AZ', 'Coconino, AZ'),  
    'Iowa': ('Polk, IA', 'Linn, IA', 'Scott, IA', 'Johnson, IA', 'Black Hawk, IA'), 
    'Kentucky': ('Jefferson, KY', 'Fayette, KY', 'Kenton, KY', 'Boone, KY', 'Warren, KY'), 
    'New Jersey': ('Bergen, NJ', 'Middlesex, NJ', 'Essex, NJ', 'Hudson, NJ', 'Monmouth, NJ'), 
    'New York': ('New York, NY', 'Kings, NY', 'Queens, NY', 'Bronx, NY', 'Richmond, NY'), 
    'North Carolina': ('Wake, NC', 'Mecklenburg, NC', 'Guilford, NC', 'Forsyth, NC', 'Cumberland, NC'), 
    'Utah': ('Salt Lake, UT', 'Utah, UT', 'Davis, UT', 'Weber, UT', 'Washington, UT')
}

catalog=RedivisCatalog(ORGNAME)
db_type="sedd"





years_icd9=[str(y) for y in range(2006, 2015)] + ["2015q1q3"]
cols_to_pull_icd9=["KEY","AGE", "ZIP", "AMONTH","AYEAR"]
prefix_icd9="ECODE"
vals_icd9=[f"E{i}" for i in range(950, 960)]

years_icd10=["2015q4"] + [str(y) for y in range(2016, 2021)]
cols_to_pull_icd10=["KEY","AGE", "ZIP", "AMONTH","AYEAR"]
prefix_icd10="I10_DX"

# 1. Suicidal Ideation and Unspecified Attempts (Exact matches)
# R45851: Suicidal ideation
# T1491xx: Suicide attempt (Initial, Subsequent, Sequela)
vals_ideation_attempt = ['R45851', 'T1491XA', 'T1491XD', 'T1491XS']

# 2. Intentional Self-Harm by mechanism (Base codes X71 through X83)
vals_self_harm_x = [f"X{i}" for i in range(71, 84)]
# 3. Intentional Self-Poisoning (Base codes T36 through T50)
vals_poisoning_t = [f"T{i}" for i in range(36, 51)]

# Combine them all into one master tuple of prefixes
# (Tuples are required withpandas .str.startswith())
vals_icd10 = tuple(vals_ideation_attempt + vals_self_harm_x + vals_poisoning_t)


In [ ]:
sedd_data = {}

for state in selected_cntys.keys():
    print(f"Fetching {state}...")
    # Fetch and enrich ICD-9
    df_icd9 = fetch_hcup(catalog, state, db_type, years_icd9, cols_to_pull_icd9, prefix_icd9, vals_icd9, return_icd_cols=False)
    df_icd9 = enrich_zip_data(df_icd9)
    
    # Fetch and enrich ICD-10
    df_icd10 = fetch_hcup(catalog, state, db_type, years_icd10, cols_to_pull_icd10, prefix_icd10, vals_icd10, return_icd_cols=False)
    df_icd10 = enrich_zip_data(df_icd10)
    
    # Store in dictionary
    sedd_data[state] = {
        'icd9': df_icd9,
        'icd10': df_icd10
    }
    # Example access for New Jersey:
# sedd_data['New Jersey']['icd9']
# sedd_data['New Jersey']['icd10']

In [ ]:
nj9=sedd_data['New Jersey']['icd9']
ny10=sedd_data['New York']['icd10']

#TODO: check the zip's NA! (could come from the function when i force them to str before conversion to county)



In [ ]:
'''
state_name = "New Jersey"
db_type="sid"
sid_nj_icd9 = fetch_hcup(catalog,state_name,db_type,years_icd9,cols_to_pull_icd9,prefix_icd9, vals_icd9, return_icd_cols=False)
sid_nj_icd9 = enrich_zip_data(sid_nj_icd9)
sid_nj_icd10 = fetch_hcup(catalog,state_name,db_type,years_icd10,cols_to_pull_icd10,prefix_icd10, vals_icd10, return_icd_cols=False)
sid_nj_icd10 = enrich_zip_data(sid_nj_icd10)
'''